# Emotion Recognition System — Dataset Quality Analysis

**Internship:** NIT Sikkim  
This notebook audits the provided YOLO dataset before preprocessing or model training. It treats `train`, `valid`, and `test` as separate splits.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import math

import cv2
import numpy as np
import pandas as pd
import yaml
from PIL import Image

pd.set_option('display.max_colwidth', None)

In [2]:
dataset_path = Path('../dataset/YOLO_format').resolve()
if not dataset_path.exists():
    dataset_path = Path('dataset/YOLO_format').resolve()

with (dataset_path / 'data.yaml').open(encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

class_names = data_config['names']
if isinstance(class_names, dict):
    class_names = [class_names[index] for index in sorted(class_names)]

splits = {
    name: {'images': dataset_path / name / 'images', 'labels': dataset_path / name / 'labels'}
    for name in ('train', 'valid', 'test')
}

print('Dataset:', dataset_path)
print('Classes:', dict(enumerate(class_names)))
for name, folders in splits.items():
    print(f"{name}: images={folders['images'].exists()}, labels={folders['labels'].exists()}")

Dataset: /Users/ankushthakur108/Nit-Sikkim-Internship/Emotion-Recognition-System/dataset/YOLO_format
Classes: {0: 'Anger', 1: 'Contempt', 2: 'Disgust', 3: 'Fear', 4: 'Happy', 5: 'Neutral', 6: 'Sad', 7: 'Surprise'}
train: images=True, labels=True
valid: images=True, labels=True
test: images=True, labels=True


## 1. Dataset structure

The configured folders and emotion-class mapping are displayed above. The following commits add image, label, duplication, and sharpness quality checks.

## 2. Image properties and resolution

Every image is decoded to check readability while recording dimensions, format, colour mode, channels, and file size.

In [3]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def image_files(folder):
    return sorted(path for path in folder.iterdir()
                  if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

def analyze_images(paths):
    result = {'total_images': len(paths), 'readable_images': 0, 'corrupt_images': 0,
              'sizes': Counter(), 'formats': Counter(), 'modes': Counter(),
              'channels': Counter(), 'file_sizes_bytes': [], 'records': [], 'errors': []}
    for path in paths:
        try:
            with Image.open(path) as image:
                image.load()
                width, height = image.size
                mode = image.mode
                channels = len(image.getbands())
                result['readable_images'] += 1
                result['sizes'][(width, height)] += 1
                result['formats'][image.format or 'Unknown'] += 1
                result['modes'][mode] += 1
                result['channels'][channels] += 1
                result['file_sizes_bytes'].append(path.stat().st_size)
                result['records'].append({'path': path, 'width': width, 'height': height,
                                          'mode': mode, 'channels': channels,
                                          'format': image.format or 'Unknown',
                                          'file_size_bytes': path.stat().st_size})
        except Exception as error:
            result['corrupt_images'] += 1
            result['errors'].append((path.name, str(error)))
    return result

split_images = {name: image_files(folders['images']) for name, folders in splits.items()}
image_analysis = {name: analyze_images(paths) for name, paths in split_images.items()}

for name, result in image_analysis.items():
    print(f'\n{name.upper()}')
    print('Total/readable/corrupt:', result['total_images'], result['readable_images'], result['corrupt_images'])
    print('Resolutions:', dict(result['sizes']))
    print('Formats:', dict(result['formats']))
    print('Modes:', dict(result['modes']))
    print('Channels:', dict(result['channels']))


TRAIN
Total/readable/corrupt: 17101 17101 0
Resolutions: {(96, 96): 17101}
Formats: {'PNG': 4934, 'JPEG': 12167}
Modes: {'RGB': 17101}
Channels: {3: 17101}

VALID
Total/readable/corrupt: 5406 5406 0
Resolutions: {(96, 96): 5406}
Formats: {'PNG': 1860, 'JPEG': 3546}
Modes: {'RGB': 5406}
Channels: {3: 5406}

TEST
Total/readable/corrupt: 2755 2755 0
Resolutions: {(96, 96): 2755}
Formats: {'PNG': 1003, 'JPEG': 1752}
Modes: {'RGB': 2755}
Channels: {3: 2755}


## 3. Image–label pairs and YOLO annotations

Every image should have a same-stem `.txt` file. Valid YOLO rows contain `class_id x_center y_center width height`, with finite normalized coordinates and positive width/height.

In [4]:
def check_pairs(image_paths, label_folder):
    image_by_stem = {path.stem: path for path in image_paths}
    label_paths = sorted(path for path in label_folder.glob('*.txt') if path.is_file())
    label_by_stem = {path.stem: path for path in label_paths}
    missing = sorted(set(image_by_stem) - set(label_by_stem))
    orphan = sorted(set(label_by_stem) - set(image_by_stem))
    return image_by_stem, label_by_stem, missing, orphan

def analyze_labels(label_paths, class_count):
    class_counts = Counter()
    invalid_rows, empty_files, annotations = [], [], 0
    for path in label_paths.values():
        nonempty_lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
        if not nonempty_lines:
            empty_files.append(path.name)
        for line_number, line in enumerate(nonempty_lines, start=1):
            fields = line.split()
            try:
                values = [float(value) for value in fields]
                class_id = int(values[0])
                valid = (len(values) == 5 and values[0].is_integer() and 0 <= class_id < class_count
                         and all(math.isfinite(value) for value in values)
                         and all(0 <= value <= 1 for value in values[1:])
                         and values[3] > 0 and values[4] > 0)
                if not valid:
                    raise ValueError('not a valid normalized YOLO row')
                class_counts[class_id] += 1
                annotations += 1
            except (ValueError, IndexError):
                invalid_rows.append((path.name, line_number, line))
    return {'annotations': annotations, 'class_counts': class_counts,
            'invalid_rows': invalid_rows, 'empty_files': empty_files}

pair_analysis, label_analysis = {}, {}
for name, folders in splits.items():
    images, labels, missing, orphan = check_pairs(split_images[name], folders['labels'])
    pair_analysis[name] = {'images': images, 'labels': labels, 'missing_labels': missing, 'orphan_labels': orphan}
    label_analysis[name] = analyze_labels(labels, len(class_names))
    print(f"\n{name.upper()}: missing labels={len(missing)}, orphan labels={len(orphan)}, empty labels={len(label_analysis[name]['empty_files'])}, invalid rows={len(label_analysis[name]['invalid_rows'])}")
    print('Annotations by emotion:', {class_names[key]: value for key, value in sorted(label_analysis[name]['class_counts'].items())})


TRAIN: missing labels=0, orphan labels=0, empty labels=0, invalid rows=0
Annotations by emotion: {'Anger': 2339, 'Contempt': 1996, 'Disgust': 2242, 'Fear': 2021, 'Happy': 2154, 'Neutral': 1616, 'Sad': 1914, 'Surprise': 2819}



VALID: missing labels=0, orphan labels=0, empty labels=0, invalid rows=0
Annotations by emotion: {'Anger': 712, 'Contempt': 618, 'Disgust': 672, 'Fear': 622, 'Happy': 791, 'Neutral': 514, 'Sad': 603, 'Surprise': 874}



TEST: missing labels=0, orphan labels=0, empty labels=0, invalid rows=0
Annotations by emotion: {'Anger': 383, 'Contempt': 332, 'Disgust': 327, 'Fear': 318, 'Happy': 399, 'Neutral': 250, 'Sad': 278, 'Surprise': 468}


## 4. Exact duplicate detection

MD5 hashes find byte-for-byte duplicate files. This checks duplicates inside each split and exact leakage between each pair of splits; it does not claim to find visually similar recompressed images.

In [5]:
def file_hash(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

hash_maps = {}
for name, paths in split_images.items():
    hashes = defaultdict(list)
    for path in paths:
        hashes[file_hash(path)].append(path)
    hash_maps[name] = hashes
    within_split = [group for group in hashes.values() if len(group) > 1]
    print(f'{name.upper()} exact duplicates within split:', sum(len(group) - 1 for group in within_split))

cross_split_duplicates = {}
split_names = list(splits)
for index, left in enumerate(split_names):
    for right in split_names[index + 1:]:
        shared_hashes = sorted(set(hash_maps[left]) & set(hash_maps[right]))
        matches = [(hash_maps[left][digest], hash_maps[right][digest]) for digest in shared_hashes]
        cross_split_duplicates[(left, right)] = matches
        print(f'Exact duplicates {left} ↔ {right}:', len(matches))

TRAIN exact duplicates within split: 126


VALID exact duplicates within split: 14


TEST exact duplicates within split: 2
Exact duplicates train ↔ valid: 66
Exact duplicates train ↔ test: 26
Exact duplicates valid ↔ test: 11


## 5. Blur / sharpness distribution

Laplacian variance is an edge-strength proxy: lower values indicate lower measured sharpness but do not prove an image is visually blurred. The lowest 10% in each split form a review queue, using a split-specific percentile threshold rather than an arbitrary cutoff.

In [6]:
def blur_score(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    return float(cv2.Laplacian(image, cv2.CV_64F).var())

sharpness = {}
for name, paths in split_images.items():
    scores = [(path, blur_score(path)) for path in paths]
    scores = [(path, score) for path, score in scores if score is not None]
    values = np.array([score for _, score in scores])
    threshold = float(np.percentile(values, 10))
    review_queue = [(path, score) for path, score in scores if score < threshold]
    sharpness[name] = {'scores': scores, 'threshold': threshold, 'review_queue': review_queue,
                       'min': float(values.min()), 'median': float(np.median(values)),
                       'max': float(values.max())}
    print(f"{name.upper()}: n={len(values)}, min={sharpness[name]['min']:.2f}, p10 threshold={threshold:.2f}, median={sharpness[name]['median']:.2f}, max={sharpness[name]['max']:.2f}, review queue={len(review_queue)}")

TRAIN: n=17101, min=351.93, p10 threshold=1564.36, median=2830.61, max=36395.61, review queue=1710


VALID: n=5406, min=443.23, p10 threshold=1580.97, median=2845.08, max=43802.11, review queue=541


TEST: n=2755, min=674.77, p10 threshold=1557.74, median=2792.19, max=18421.48, review queue=276


## 6. Dataset quality summary

The low-sharpness figure is a relative 10th-percentile grouping, not a definite count of blurry images. Inspect the lowest-scoring files before excluding or altering them.

In [7]:
summary_rows = []
for name in splits:
    result = image_analysis[name]
    file_sizes = result['file_sizes_bytes']
    summary_rows.append({
        'Split': name, 'Images': result['total_images'], 'Readable': result['readable_images'],
        'Corrupt': result['corrupt_images'], 'Unique resolutions': len(result['sizes']),
        'Resolutions': ', '.join(f'{width}×{height} ({count})' for (width, height), count in result['sizes'].items()),
        'Formats': dict(result['formats']), 'Modes': dict(result['modes']),
        'Median file size (KiB)': round(float(np.median(file_sizes)) / 1024, 2),
        'Missing labels': len(pair_analysis[name]['missing_labels']),
        'Orphan labels': len(pair_analysis[name]['orphan_labels']),
        'Empty labels': len(label_analysis[name]['empty_files']),
        'Invalid annotation rows': len(label_analysis[name]['invalid_rows']),
        'Annotations': label_analysis[name]['annotations'],
        'Low-sharpness review (lowest 10%)': len(sharpness[name]['review_queue']),
        'Sharpness p10 threshold': round(sharpness[name]['threshold'], 2),
    })
summary = pd.DataFrame(summary_rows)
summary

,Split,Images,Readable,Corrupt,Unique resolutions,Resolutions,Formats,Modes,Median file size (KiB),Missing labels,Orphan labels,Empty labels,Invalid annotation rows,Annotations,Low-sharpness review (lowest 10%),Sharpness p10 threshold
0,train,17101,17101,0,1,96×96 (17101),"{'PNG': 4934, 'JPEG': 12167}",{'RGB': 17101},6.18,0,0,0,0,17101,1710,1564.36
1,valid,5406,5406,0,1,96×96 (5406),"{'PNG': 1860, 'JPEG': 3546}",{'RGB': 5406},6.31,0,0,0,0,5406,541,1580.97
2,test,2755,2755,0,1,96×96 (2755),"{'PNG': 1003, 'JPEG': 1752}",{'RGB': 2755},6.33,0,0,0,0,2755,276,1557.74


In [8]:
print('Examples for manual review: lowest measured-sharpness images in each split')
for name in splits:
    examples = sorted(sharpness[name]['scores'], key=lambda item: item[1])[:10]
    print(f'\n{name.upper()}')
    for path, score in examples:
        print(f'{path.name}: {score:.2f}')

print('\nInvalid annotation examples (if any):')
for name in splits:
    print(name, label_analysis[name]['invalid_rows'][:5])

Examples for manual review: lowest measured-sharpness images in each split

TRAIN
ffhq_3670.png: 351.93
ffhq_4114.png: 420.16
ffhq_5153.png: 484.40
ffhq_5048.png: 505.94
ffhq_4645.png: 532.70
ffhq_4022.png: 543.59
ffhq_4574.png: 550.07
image0037830.jpg: 562.31
ffhq_1877.png: 593.26
ffhq_398.png: 605.35

VALID
image0031845.jpg: 443.23
ffhq_3120.png: 459.33
ffhq_1333.png: 600.82
image0031577.jpg: 642.87
image0022731.jpg: 685.71
ffhq_2209.png: 692.64
ffhq_4085.png: 699.09
ffhq_2332.png: 715.31
image0031544.jpg: 735.50
ffhq_4063.png: 764.79

TEST
image0031573.jpg: 674.77
ffhq_2027.png: 713.23
image0021390.jpg: 744.30
ffhq_5489.png: 770.35
image0041525.jpg: 773.08
ffhq_3018.png: 789.26
image0035606.jpg: 812.05
ffhq_78.png: 846.45
ffhq_5426.png: 864.00
image0001819.jpg: 874.92

Invalid annotation examples (if any):
train []
valid []
test []


## 7. Evidence-based preprocessing plan

1. Resolve any corrupt files, missing/orphan labels, or invalid annotations before training; record each decision.
2. Manually inspect the low-sharpness review queue. Retain naturally low-detail but correctly labelled faces unless visual review shows they are unusable.
3. If a common model input size is needed, resize in the data-loader with a documented interpolation method. Preserve aspect ratio with letterboxing when required.
4. Resizing or upscaling increases pixel dimensions but cannot recover facial detail that was not captured. Do not claim it creates genuine high-resolution information.
5. Evaluate any enhancement or super-resolution method in an ablation: apply it consistently, train comparable models, and compare held-out metrics. Do not fit preprocessing on validation or test images.
6. Keep train, validation, and test splits isolated. Remove or reassign cross-split exact duplicates before reporting final evaluation metrics.